In [ ]:

# DECISIVE RUN. A three-seed pilot put the dilated, mask-source-matched pipeline
# at 83.90 against the standalone classifier's 83.61. This paper's own argument
# is that three seeds neither estimate an effect nor its precision, so that
# pilot cannot be reported as a result. Here all three regimes are run at TEN
# seeds on BOTH test sets.
#   A  classifier trained on ground-truth masks, tested on predicted   (published)
#   B  trained and tested on predicted masks                            (matched)
#   C  trained and tested on dilated predicted masks, keeping a rim of
#      peri-lesional skin                                               (matched + context)
import subprocess, sys, os, json, glob, time
info = subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("GPU:", info or "NONE"); cap = info.split(",")[1].strip() if "," in info else ""
assert "torch" not in sys.modules
if cap.startswith("6."):
    print(f"compute capability {cap} is Pascal: pinning torch 2.5.1 + cu121")
    subprocess.run([sys.executable,"-m","pip","-q","install","torch==2.5.1","torchvision==0.20.1",
                    "--index-url","https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run([sys.executable,"-m","pip","-q","uninstall","-y",
                "ray","wandb","comet_ml","mlflow","dvclive","neptune","clearml"], check=False)
subprocess.run([sys.executable,"-m","pip","-q","install","ultralytics==8.3.40","timm==1.0.11"], check=True)
import torch
print("torch", torch.__version__, "| arch", torch.cuda.get_arch_list())
assert torch.cuda.is_available()
_x=(torch.randn(64,64,device="cuda")@torch.randn(64,64,device="cuda")).sum().item(); torch.cuda.synchronize()
print("CUDA smoke PASSED", round(_x,3))


In [ ]:

import numpy as np, cv2
from ultralytics import YOLO
UN=None
for dp,dns,fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("unmasked") and {"train","valid","test"}<=set(dns): UN=dp; break
GT=None
for dp,dns,fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("masked") and {"train","valid","test"}<=set(dns) and "unmasked" not in dp: GT=dp; break
EXT=None
for dp,dns,fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("images") and "external" in dp: EXT=dp; break
LOC=glob.glob("/kaggle/input/**/*1class_LEAKFREE.pt", recursive=True)[0]
print("unmasked:",UN,"\nground-truth masked:",GT,"\nexternal:",EXT,"\nlocaliser:",LOC)

m=YOLO(LOC); OUT="/kaggle/working/pred"
K=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(25,25))
def masks_for(img,p):
    H,W=img.shape[:2]
    r=m.predict(p,conf=0.05,verbose=False)[0]
    if r.masks is None or len(r.masks.data)==0: mk=np.ones((H,W),np.uint8)
    else:
        mm=r.masks.data.cpu().numpy().max(0)
        mk=(cv2.resize(mm,(W,H),interpolation=cv2.INTER_NEAREST)>0.5).astype(np.uint8)
    return mk, cv2.dilate(mk,K,iterations=1)

t0=time.time(); n=0
for split in ("train","valid","test"):
    for cls in sorted(os.listdir(f"{UN}/{split}")):
        d=f"{UN}/{split}/{cls}"
        if not os.path.isdir(d): continue
        for reg in ("hard","dilated"): os.makedirs(f"{OUT}/{reg}/{split}/{cls}",exist_ok=True)
        for p in sorted(glob.glob(d+"/*")):
            img=cv2.imread(p)
            if img is None: continue
            mk,dl=masks_for(img,p); b=os.path.basename(p)
            cv2.imwrite(f"{OUT}/hard/{split}/{cls}/{b}", img*mk[...,None])
            cv2.imwrite(f"{OUT}/dilated/{split}/{cls}/{b}", img*dl[...,None])
            n+=1
            if n%600==0: print(f"  internal {n} ({(time.time()-t0)/60:.1f} min)",flush=True)

# external set: ground truth comes from the released per-image predictions
gtf=glob.glob("/kaggle/input/**/preds_external.json", recursive=True)
extgt={r["img"]:r["true"] for r in json.load(open(gtf[0]))} if gtf else {}
print("external labels:",len(extgt))
for reg in ("hard","dilated"):
    for c in ("0","1","2"): os.makedirs(f"{OUT}/{reg}/ext/{c}",exist_ok=True)
ne=0
for p in sorted(glob.glob(EXT+"/*")):
    b=os.path.basename(p)
    if b not in extgt: continue
    img=cv2.imread(p)
    if img is None: continue
    mk,dl=masks_for(img,p); c=str(extgt[b])
    cv2.imwrite(f"{OUT}/hard/ext/{c}/{b}", img*mk[...,None])
    cv2.imwrite(f"{OUT}/dilated/ext/{c}/{b}", img*dl[...,None])
    ne+=1
print(f"generated internal={n} external={ne} in {(time.time()-t0)/60:.1f} min")


In [ ]:

import numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
IMG=224
tf_tr=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.RandomHorizontalFlip(),
  transforms.ColorJitter(0.2,0.2,0.2),transforms.RandomRotation(10),transforms.ToTensor(),
  transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_ev=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.ToTensor(),
  transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
PRED="/kaggle/working/pred"

def bal(P,L):
    per={}
    for a,b in zip(P,L):
        per.setdefault(b,[0,0]); per[b][1]+=1
        if a==b: per[b][0]+=1
    return 100*float(np.mean([c/t for c,t in per.values()]))

def run(train_root, test_root, ext_root, seed, epochs=30, patience=7, batch=16, lr=1e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    tr=datasets.ImageFolder(f"{train_root}/train",tf_tr); va=datasets.ImageFolder(f"{train_root}/valid",tf_ev)
    te=datasets.ImageFolder(f"{test_root}/test",tf_ev);   ex=datasets.ImageFolder(ext_root,tf_ev)
    cnt=np.bincount([y for _,y in tr.samples],minlength=3).astype(float)
    w=torch.tensor(cnt.sum()/(3*cnt),dtype=torch.float32).cuda()
    tl=DataLoader(tr,batch_size=batch,shuffle=True,num_workers=2)
    vl=DataLoader(va,batch_size=64,num_workers=2); el=DataLoader(te,batch_size=64,num_workers=2)
    xl=DataLoader(ex,batch_size=64,num_workers=2)
    mdl=timm.create_model("swin_tiny_patch4_window7_224",pretrained=True,num_classes=3).cuda()
    opt=torch.optim.AdamW(mdl.parameters(),lr=lr,weight_decay=0.05)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    crit=nn.CrossEntropyLoss(weight=w); sc=torch.amp.GradScaler("cuda")
    best,bad,st=-1,0,None
    for ep in range(epochs):
        mdl.train()
        for x,y in tl:
            x,y=x.cuda(non_blocking=True),y.cuda(non_blocking=True); opt.zero_grad()
            with torch.amp.autocast("cuda"): loss=crit(mdl(x),y)
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        sch.step(); mdl.eval(); c=t=0
        with torch.no_grad():
            for x,y in vl:
                with torch.amp.autocast("cuda"): p=mdl(x.cuda()).argmax(1).cpu()
                c+=(p==y).sum().item(); t+=len(y)
        v=100*c/t
        if v>best: best,bad,st=v,0,{k:q.detach().clone() for k,q in mdl.state_dict().items()}
        else:
            bad+=1
            if bad>=patience: break
    mdl.load_state_dict(st); mdl.eval(); out={}
    for tag,dl in (("int",el),("ext",xl)):
        P=[];L=[]
        with torch.no_grad():
            for x,y in dl:
                with torch.amp.autocast("cuda"): P+=mdl(x.cuda()).argmax(1).cpu().tolist()
                L+=y.tolist()
        out[tag+"_acc"]=100*sum(int(a==b) for a,b in zip(P,L))/len(L)
        out[tag+"_bal"]=bal(P,L); out[tag+"_preds"]=P; out[tag+"_labels"]=L
    out["val"]=best; return out

GT=None
for dp,dns,fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("masked") and {"train","valid","test"}<=set(dns) and "unmasked" not in dp: GT=dp; break
ARMS=[("A_gt_train_pred_test", GT, f"{PRED}/hard", f"{PRED}/hard/ext"),
      ("B_matched_hard", f"{PRED}/hard", f"{PRED}/hard", f"{PRED}/hard/ext"),
      ("C_matched_dilated", f"{PRED}/dilated", f"{PRED}/dilated", f"{PRED}/dilated/ext")]
res=[]
for nm,trr,ter,exr in ARMS:
    for s in range(10):
        t0=time.time(); r=run(trr,ter,exr,s); r.update(arm=nm,seed=s,minutes=round((time.time()-t0)/60,1))
        res.append(r); json.dump(res,open("/kaggle/working/pipeline_10seed_regimes.json","w"),indent=1)
        print(f"  {nm:22} seed {s}: int {r['int_acc']:6.2f}  ext {r['ext_acc']:6.2f}  ({r['minutes']}m)",flush=True)

print("\n=== TEN-SEED SUMMARY ===")
for nm,_,_,_ in ARMS:
    a=np.array([r["int_acc"] for r in res if r["arm"]==nm])
    e=np.array([r["ext_acc"] for r in res if r["arm"]==nm])
    print(f"  {nm:22} internal {a.mean():6.2f} +- {a.std(ddof=1):4.2f}   external {e.mean():6.2f} +- {e.std(ddof=1):4.2f}")
print("\n  BASELINE standalone ConvNeXt-Large, ten seeds: internal 83.61 +- 1.74, external 77.81 +- 2.29")
